# A Deep Agents demo

In this notebook, we'll build an agent that works with a folder of meeting notes and answers questions like *is the 30 September beta on track?* It has to find the notes itself, work out the answer from facts spread across them, and write that answer to a file.

First we'll use a plain Python loop, like in the previous modules, and then we'll translate it to `deepagents`. Finally we add a fact-checker subagent, demonstrating how subagents work.

## 0 — Setup

The model runs on Nebius Token Factory, so don't forget the API key. The cell reads it from `.env`, falls back to the `token-factory-key` file in this folder, and fails immediately if neither is there.

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("NEBIUS_API_KEY") and os.path.exists("token-factory-key"):
    os.environ["NEBIUS_API_KEY"] = open("token-factory-key", encoding="utf-8").read().strip()

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in .env"
print("Keys loaded.")

Keys loaded.


### The data

Let's create three meeting notes from a fictional project called Atlas,
written to `workspace/notes/` by the cell below. Each is dated and has the same
three sections: progress, decisions, open questions.

- **`2026-08-14-kickoff.md`** — sets the 30 September beta date, and leaves two
  questions open: who owns on-call, and whether legal has to review indexing
  tickets that contain customer names.
- **`2026-08-21-standup.md`** — the first retrieval score, 61%, against a bar of
  80%. Legal review is confirmed as required. A checkpoint is set for
  10 September: if quality is still under 80% by then, scope shrinks to
  search-only.
- **`2026-08-28-review.md`** — retrieval is up to 76%, legal signed off on
  26 Aug, and the checkpoint stands.

So no single note answers "are we on track", and the agent will have to gather the information piece by piece..

In [4]:
from pathlib import Path

WORKSPACE = Path.cwd() / "workspace"
(WORKSPACE / "notes").mkdir(parents=True, exist_ok=True)

NOTES = {
    "2026-08-14-kickoff.md": """# Atlas kickoff — 14 Aug 2026

Present: product, design, two backend engineers, one ML engineer.

## What we're building
"Atlas" — an internal assistant that answers questions over the company's
support tickets. Read-only for v1: no ticket edits, no customer-facing replies.

## Decisions
- **Scope v1 to search + summarize.** No ticket writes until we have an audit log.
- **Ship to the support team first** (12 people), not the whole company.
- **Target date: 30 September 2026** for the internal beta.
- Vector store: we go with the managed option rather than self-hosting, to save
  the ML engineer's time for retrieval quality work.

## Open questions
- Who owns the on-call rotation once Atlas is in front of the support team?
- Do we need legal review before indexing tickets that contain customer names?
""",
    "2026-08-21-standup.md": """# Atlas weekly standup — 21 Aug 2026

## Progress
- Ticket ingestion pipeline runs nightly; 340k tickets indexed.
- First retrieval eval: 61% of answers cite the correct ticket. Below the 80%
  bar we set informally at kickoff.

## Decisions
- **Add a re-ranking step** before we touch the embedding model. Cheaper to try.
- **Legal review is required** — confirmed with the legal team. We will strip
  customer names at ingestion time rather than wait for a policy exemption.
- Beta date stays 30 September, but scope may shrink to search-only if
  retrieval quality is still under 80% by 10 September.

## Open questions
- On-call ownership is still unassigned (carried over from kickoff).
- Is 80% the right bar? Nobody has written down how it was chosen.
""",
    "2026-08-28-review.md": """# Atlas review — 28 Aug 2026

## Progress
- Re-ranking landed. Retrieval eval now at 76%, up from 61%.
- Name stripping is live in the ingestion pipeline; legal signed off on 26 Aug.

## Decisions
- **Support team owns on-call**, with an ML engineer as secondary for the first
  month. This closes the question raised at kickoff.
- **The 80% bar is now formal** and written into the beta checklist.
- **We will not shrink scope yet** — the 10 September checkpoint stands.

## Open questions
- Do we need a feedback button in the UI for the beta, or is a shared channel
  enough for 12 users?
- Nobody has costed what happens if usage grows past the support team.
""",
}

for name, body in NOTES.items():
    (WORKSPACE / "notes" / name).write_text(body, encoding="utf-8")


## 1. The agent as a Python loop

### The tools

We'll use three functions that work with paths relative to `workspace/`.

- **`list_files(path=".")`** — returns one line per entry in a directory: name
  and size for files, a trailing `/` for folders.
- **`read_file(path)`** — returns the text content of a file, cut off at
  `MAX_READ_CHARS` (20,000) so one oversized file cannot swallow the context
  window.
- **`write_file(path, content)`** — writes the file and returns a confirmation
  string. This is how the agent delivers its answer.
- **`_resolve(path)`** — joins the path onto `workspace/`, resolves it, and
  raises `ValueError` if the result lands outside. The other three call it
  before touching the disk.

We need `_resolve` because we want to prevent the agent from accessing anything
outside the `workspace/` folder. The three tools catch that error and return its
message, so the model is told "no" as an ordinary tool result and can try
another path, instead of the exception ending the run.

If the model queries for a file outside the `workspace/` folder, it will get a `Path escapes the workspace` message.

In [5]:
MAX_READ_CHARS = 20_000


def _resolve(path: str) -> Path:
    """Resolve a model-supplied path inside the workspace, or refuse it.

    The three tools catch this and hand the message back to the model as the
    tool result, so a bad path costs one step instead of killing the run.
    """
    target = (WORKSPACE / path).resolve()
    if target != WORKSPACE and WORKSPACE not in target.parents:
        raise ValueError(f"Path escapes the workspace: {path}")
    return target

def list_files(path: str = ".") -> str:
    """List files and directories at `path`, relative to the workspace root."""
    try:
        target = _resolve(path)
    except ValueError as exc:
        return str(exc)
    if not target.is_dir():
        return f"Not a directory: {path}"
    entries = []
    for child in sorted(target.iterdir()):
        if child.name.startswith(".") or child.name == "__pycache__":
            continue
        rel = child.relative_to(WORKSPACE).as_posix()
        entries.append(f"{rel}/" if child.is_dir() else f"{rel} ({child.stat().st_size} bytes)")
    return "\n".join(entries) if entries else f"(empty directory: {path})"

def read_file(path: str) -> str:
    """Read a text file and return its content (truncated if very long)."""
    try:
        target = _resolve(path)
    except ValueError as exc:
        return str(exc)
    if not target.is_file():
        return f"No such file: {path}"
    content = target.read_text(encoding="utf-8", errors="replace")
    if len(content) > MAX_READ_CHARS:
        return content[:MAX_READ_CHARS] + f"\n... [truncated at {MAX_READ_CHARS} chars]"
    return content

def write_file(path: str, content: str) -> str:
    """Write `content` to `path`, creating parent directories as needed."""
    try:
        target = _resolve(path)
    except ValueError as exc:
        return str(exc)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Wrote {len(content)} chars to {path}"

`AGENT_TOOLS` is what gets sent to the model with every request; `TOOLS` maps a
name back to the function, so the loop can run whatever the model asks for.

In [ ]:
AGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List the files and folders at a path in the workspace. Call this first to find out what exists.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Directory relative to the workspace root. Defaults to '.'."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read one text file from the workspace and return its content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "File path relative to the workspace root, e.g. 'notes/2026-08-28-review.md'."
                    }
                },
                "required": [
                    "path"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write a text file into the workspace, overwriting it if it exists. Use this to deliver your final document.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "File path relative to the workspace root, e.g. 'go-no-go.md'."
                    },
                    "content": {
                        "type": "string",
                        "description": "Full file content to write."
                    }
                },
                "required": [
                    "path",
                    "content"
                ]
            }
        }
    }
]

TOOLS = {
    "list_files": list_files,
    "read_file": read_file,
    "write_file": write_file,
}

### The model and the client

In [ ]:
import json
from datetime import datetime
from typing import Callable

from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

MODEL = "nvidia/nemotron-3-super-120b-a12b"

### The instructions

Note that we append today's date to the system prompt, because the notes are
dated and the question might be about a deadline.

In [ ]:
LOOP_SYSTEM_PROMPT = """You are a careful analyst working inside a small file workspace.

You have three tools: list_files, read_file and write_file. Paths are relative
to the workspace root, e.g. `notes/2026-08-28-review.md`.

How to work:
1. Call list_files first to see what is there before assuming any filename.
2. Read every relevant file before writing anything -- do not summarise a file
   you have not read.
3. When a later note contradicts an earlier one, the later note wins; say so.
4. Deliver your answer by calling write_file, once. Do not paste the document
   into the chat instead of writing it.
5. After write_file succeeds, reply with one short sentence saying what you
   wrote and where. Do not call more tools.

Be concrete: numbers, dates and filenames, not vague summary language."""

SYSTEM_PROMPT = f"""{LOOP_SYSTEM_PROMPT} Today is {datetime.now().strftime("%Y-%m-%d")}"""

### The loop

`run_agent` keeps a list of messages and repeats three steps:

1. send the messages and the tool schemas to the model,
2. if the reply contains tool calls, run each one and append its result as a
   `tool` message, then go round again,
3. if the reply is plain text instead, that text is the answer.

`max_steps` bounds the number of rounds. On this task, the trace is usually six
steps: `list_files` once, `read_file` three times, `write_file` once, then a
closing sentence.

In [ ]:
def run_agent(
    question: str,
    model: str = MODEL,
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=AGENT_TOOLS
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"Step {step}: final answer")
            return msg.content

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                # Truncated: write_file's argument is a whole document.
                shown = str(args)
                print(f"Step {step}: {tc.function.name} {shown[:120]}{'...' if len(shown) > 120 else ''}")
            result = TOOLS[tc.function.name](**args)
            if verbose:
                print(f"Result snippet: {result[:100]} ... {result[-100:]}")
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )

    return "Stopped: reached max_steps without a final answer."

The question we put to the agent, and the run:

In [ ]:
TASK = """The Atlas beta is set for 30 September.
Read the notes in notes/ and write go-no-go.md answering one question: are we on track?

Cover:
- where retrieval quality stands: how many points it has gained since the first
  eval, and how far it still is from the bar,
- what was a blocker and has since been cleared,
- what has to happen by the 10 September checkpoint.

Report the state as of the latest note, and cite the note filename for every point."""

answer = run_agent(TASK)
print()
print("Answer:", answer)

Step 1: list_files {'path': 'notes/'}
Result snippet: notes/2026-08-14-kickoff.md (834 bytes)
notes/2026-08-21-standup.md (780 bytes)
notes/2026-08-28-rev ... ickoff.md (834 bytes)
notes/2026-08-21-standup.md (780 bytes)
notes/2026-08-28-review.md (683 bytes)


Step 2: read_file {'path': 'notes/2026-08-14-kickoff.md'}
Result snippet: # Atlas kickoff — 14 Aug 2026

Present: product, design, two backend engineers, one ML engineer.

## ... of the support team?
- Do we need legal review before indexing tickets that contain customer names?



Step 3: read_file {'path': 'notes/2026-08-21-standup.md'}
Result snippet: # Atlas weekly standup — 21 Aug 2026

## Progress
- Ticket ingestion pipeline runs nightly; 340k tic ... ned (carried over from kickoff).
- Is 80% the right bar? Nobody has written down how it was chosen.



Step 4: read_file {'path': 'notes/2026-08-28-review.md'}
Result snippet: # Atlas review — 28 Aug 2026

## Progress
- Re-ranking landed. Retrieval eval now at 76%, up from 61 ... annel
  enough for 12 users?
- Nobody has costed what happens if usage grows past the support team.



Step 5: write_file {'path': 'go-no-go.md', 'content': '# Atlas Go/No-Go Assessment (as of 2026-08-28)\n\n**Are we on track?**  \nYes – we a...
Result snippet: Wrote 1071 chars to go-no-go.md ... Wrote 1071 chars to go-no-go.md


Step 6: final answer

Answer: 

I wrote the go/no-go assessment to go-no-go.md.


As suggested in the system prompt, the agent writes the answer to a file inside
the `workspace/` folder. In this example, it's likely `workspace/go-no-go.md`.

In [ ]:
print(read_file("go-no-go.md"))

# Atlas Go/No-Go Assessment (as of 2026-08-28)

**Are we on track?**  
Yes – we are making steady progress toward the retrieval quality target, the legal blocker has been cleared, and we understand what must be achieved by the 10 September checkpoint.

## Retrieval quality
- **Gain since first eval:** 15 percentage points (from 61% to 76%)【notes/2026-08-21-standup.md】→【notes/2026-08-28-review.md】  
- **Distance to bar:** 4 percentage points below the 80% target【notes/2026-08-28-review.md】

## Blocker cleared
- **Legal review / name stripping:** Identified as a blocker in the 21 Aug standup; completed and signed off by legal on 26 Aug, with name stripping live in the ingestion pipeline【notes/2026-08-21-standup.md】→【notes/2026-08-28-review.md】

## What must happen by the 10 September checkpoint
- **Retrieval quality ≥ 80%** to avoid scope reduction to search‑only; the checkpoint stands and we will not shrink scope yet if we meet this threshold【notes/2026-08-21-standup.md】→【notes/2026-08-

Feel free to check if it's correct or not.

## 2. The same agent on `deepagents`

The `deepagents` library replaces the loop, the message bookkeeping and the tool
orchestration with library code. We keep our three functions and hand them over
as they are:

| written by hand above | in `deepagents` |
|---|---|
| the `for step in range(max_steps)` body | set up by `create_deep_agent(...)` |
| the `TOOLS` name-to-function table | `tools=[list_files, read_file, write_file]` |
| the `AGENT_TOOLS` schemas | generated from the functions themselves |
| `messages.append({"role": "tool", ...})` | handled inside |
| `max_steps=10` | `recursion_limit` on `invoke` |
| the `_resolve` path check | still ours — it lives inside our tools |
| a call of `run_agent` | calling `agent.invoke(...)` |

The prompt does not change either: same three tools, same relative paths.

In [ ]:
from langchain_nebius import ChatNebius

CHAT_MODEL = ChatNebius(
    model=MODEL,
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

In [ ]:
DEEP_AGENT_PROMPT = """You are a careful analyst working inside a small file workspace.

You have three tools: list_files, read_file and write_file. Paths are relative
to the workspace root, e.g. `notes/2026-08-28-review.md`.

How to work:
1. Call list_files first to see what is there before assuming any filename.
2. Read every relevant file before writing anything -- do not summarise a file
   you have not read.
3. When a later note contradicts an earlier one, the later note wins; say so.
4. Deliver your answer by calling write_file, once. Do not paste the document
   into the chat instead of writing it.
5. After write_file succeeds, reply with one short sentence saying what you
   wrote and where. Do not call more tools.

Be concrete: numbers, dates and filenames, not vague summary language."""

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

# Setting up the agent
agent = create_deep_agent(
    model=CHAT_MODEL,
    system_prompt=DEEP_AGENT_PROMPT,
    tools=[list_files, read_file, write_file],
    middleware=[TodoListMiddleware()],
)

# Calling the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": TASK}]},
    {"recursion_limit": 40},
)

step = 0
for message in result["messages"]:
    for tool_call in getattr(message, "tool_calls", None) or []:
        step += 1
        print(f"Step {step}: {tool_call['name']} {str(tool_call['args'])[:100]}...")

print()
print("Answer:", result["messages"][-1].content)

Step 1: list_files {'path': 'notes/'}...
Step 2: read_file {'path': 'notes/2026-08-14-kickoff.md'}...
Step 3: read_file {'path': 'notes/2026-08-21-standup.md'}...
Step 4: read_file {'path': 'notes/2026-08-28-review.md'}...
Step 5: write_file {'path': 'go-no-go.md', 'content': '# Atlas Go/No-Go Assessment\n\n**As of the latest note (notes/20...

Answer: 
Wrote go-no-go.md in the current directory.


### Where the tool descriptions come from

We passed the functions and wrote no schemas, but the model still has to be told
what each tool does and what arguments it takes. LangChain reads that off the
function itself:

- the **docstring** becomes the tool's description,
- the **type hints** become the argument types,
- a **default value** makes an argument optional.

So `def read_file(path: str) -> str:` with its one-line docstring produces the
same thing we wrote by hand in `AGENT_TOOLS`. Run the cell to see it.

Two consequences worth knowing:

- A function with **no docstring is rejected**: LangChain raises
  `ValueError: Function must have a docstring if description not provided.`
- Per-argument descriptions (`"e.g. 'notes/2026-08-28-review.md'"` in our
  hand-written schema) are *not* generated by default. To keep them, write an
  `Args:` block in the docstring and wrap the function with
  `@tool(parse_docstring=True)`.

In [ ]:
from langchain_core.tools import tool as make_tool

as_tool = make_tool(read_file)
print("description:", as_tool.description)
print("arguments  :", as_tool.args)

description: Read a text file and return its content (truncated if very long).
arguments  : {'path': {'title': 'Path', 'type': 'string'}}


## `create_deep_agent`, argument by argument

That call again, in full:

```python
agent = create_deep_agent(
    model=CHAT_MODEL,
    system_prompt=DEEP_AGENT_PROMPT,
    tools=[list_files, read_file, write_file],
    middleware=[TodoListMiddleware()],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": TASK}]},
    {"recursion_limit": 40},
)

result["messages"][-1].content
```

- **`model`** — the `ChatNebius` instance built above.
- **`system_prompt`** — our instructions. `deepagents` adds its own text on top
  of it.
- **`tools`** — our three functions, passed as they are.
- **`middleware`** — how extra tools get attached to the agent.
  `TodoListMiddleware()` brings one: `write_todos`, which the agent uses to keep
  a plan across a long job. It is not in the default set, so without this line
  the agent has no planning tool. Other middleware change how the existing tools
  behave instead of adding one — `HumanInTheLoopMiddleware`, for example, makes
  the agent stop and wait for a person to approve a tool call before it runs.
- **`invoke`'s second argument** — `recursion_limit` is the step budget, the job
  `max_steps` did in the loop. It defaults to 25 whether you pass it or not.
- **the result** — `result["messages"]` is the whole trajectory: every tool call
  and every tool result, which is what the trace above walked through. The last
  message is the answer.

### In this particular example, you can skip writing the tools entirely

`deepagents` ships its own file tools — `ls`, `read_file`, `write_file`,
`edit_file`, `delete`, `glob`, `grep` — and you get them by passing a **backend**
instead of a `tools` list:

```python
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model=CHAT_MODEL,
    system_prompt=SOME_PROMPT,
    backend=FilesystemBackend(root_dir=str(WORKSPACE), virtual_mode=True),
)
```

The backend says where those files physically live:

- `FilesystemBackend(root_dir=..., virtual_mode=True)` — sets up a folder inside which the agent will be working.
  `virtual_mode=True` does the job of our `_resolve`: it shows the folder to the
  agent as `/` (so `workspace/notes/` becomes `/notes/`) and refuses paths that
  climb above it. With `virtual_mode=False` there is no containment at all.
- `StateBackend` — files live in memory and vanish with the conversation. This
  is what you get if you pass neither a backend nor tools.
- `StoreBackend(namespace=...)` — LangGraph's store, so files survive across
  conversations.

That is less code, and it buys you `edit_file` (patch a file instead of
rewriting it), `glob` and `grep`. We use our own tools here because you already
have them from section 1, and because a real project's tools are rarely just
file access.

Two things to know if you mix the two: a tool of yours whose name matches a
built-in (like `read_file`) replaces the built-in, and the backend has no effect
on your own functions.

## 3. Adding subagents

While we have just one agent, switching to `deepagents` is only maginally effective. Agentic frameworks start paying off when you have to deal with the whole agentic systems and complex interactions inside them.

As a simple example, we'll give our agents a subagent - a fact-checker that re-reads the notes and audits the draft.

The lead's prompt changes accordingly — draft `go-no-go.md`, call the
fact-checker, apply whatever it returns — and the call happens through a `task`
tool that `deepagents` adds as soon as there is a subagent to call. The subagent
itself is a smilar dict:

```python
fact_checker_subagent = {
    "name": "fact-checker",
    "description": "Use this AFTER writing the draft and BEFORE replying. ...",
    "system_prompt": FACT_CHECKER_PROMPT,
    "tools": [list_files, read_file],
}
```

- **`name`** — what the lead passes to `task`.
- **`description`** — the text the lead reads when deciding whether to delegate,
  and what to send.
- **`system_prompt`** — the checker's own instructions: read the draft, read the
  notes, report wrong numbers, wrong dates etc.
- **`tools`** — its own list, which might be totally different from the lead's tools if task requires so. In our example, it gets `list_files` and `read_file` but not `write_file`, so it cannot edit the document it is judging: it reports, and the lead applies the fix. If you leave this key out, the subagent inherits the lead's tools.

By default, the subagent runs in `mode="isolated"`, which means its conversation starts empty. It doesn't see lead's own conversation; only what the lead communicates in the subagent's prompt. In particular, the fact-checker will have to independently read the notes.

The default way of communication between a lead and a subagent is in messages:

- the lead sends a prompt to its subagent,
- the subagent's final message os delivered to the lead as an ordinary `ToolMessage` for the task call.

However, in the interest of reliability and easier debugging, it might be good to prompt them to communicate through files.

In [ ]:
LEAD_AGENT_PROMPT = """You are a careful analyst working inside a small file workspace.

You have three tools: list_files, read_file and write_file. Paths are relative
to the workspace root, e.g. `notes/2026-08-28-review.md`.

How to work:
1. Call list_files first, then read every relevant note before writing anything.
2. When a later note contradicts an earlier one, the later note wins; say so.
3. Write your draft to the file you were asked for, once.
4. Then delegate to the `fact-checker` subagent, ONCE, before you reply. It
   starts with a blank context and cannot see this conversation, so your task
   instruction must name (a) the exact path of the draft you wrote and (b) the
   folder holding the source notes.
5. Apply every correction it returns by calling write_file again with the fixed
   document. If it returns none, change nothing.
6. Reply with one short sentence: what you wrote, where, and what the check
   found. Do not call more tools.

Be concrete: numbers, dates and filenames, not vague summary language."""

In [ ]:
FACT_CHECKER_PROMPT = """You audit a draft document against the notes it was written from.

You can read; you cannot write. Do not attempt to fix the document -- report,
and let the lead agent fix it.

Read the draft, then read every note in the notes folder. For each factual claim
in the draft, check:

- **Numbers.** Percentages, point differences and counts must match the notes,
  including any arithmetic (a "gained N points" claim must equal the difference
  between the two figures actually recorded).
- **Dates.** Deadlines and checkpoints must match what the notes say.
- **Staleness.** This is the one that bites: a question opened in an early note
  and answered in a later one must NOT be reported as still open or still
  blocking. Check the latest note before believing any "outstanding" claim.
- **Attribution.** A claim cited to a note must actually appear in that note.

Return either the exact string `No corrections.` or a numbered list. Each item:
the wrong text quoted from the draft, what the notes actually say, and the note
filename. Be specific enough that the lead can apply the fix without re-reading
everything. Do not pad the list -- a correct draft gets `No corrections.`"""

In [ ]:
fact_checker_subagent = {
    "name": "fact-checker",
    "description": (
        "Use this AFTER writing the draft and BEFORE replying. Your task instruction MUST "
        "include (1) the exact path of the draft to audit and (2) the folder holding the "
        "source notes -- it starts with an empty context and can see neither. It returns "
        "`No corrections.` or a numbered list of wrong numbers, dates, and claims that a "
        "later note has already superseded."
    ),
    "system_prompt": FACT_CHECKER_PROMPT,
    "tools": [list_files, read_file],
}

lead = create_deep_agent(
    model=CHAT_MODEL,
    system_prompt=LEAD_AGENT_PROMPT,
    tools=[list_files, read_file, write_file],
    subagents=[fact_checker_subagent],
    middleware=[TodoListMiddleware()],
)

In [ ]:
result = lead.invoke(
    {"messages": [{"role": "user", "content": TASK}]},
    {"recursion_limit": 40},
)

step = 0
for message in result["messages"]:
    for tool_call in getattr(message, "tool_calls", None) or []:
        step += 1
        if tool_call["name"] == "task":
            print(f"Step {step}: task -> {tool_call['args'].get('subagent_type')}")
            print(f"  {str(tool_call['args'].get('description'))[:140]}...")
        else:
            print(f"Step {step}: {tool_call['name']} {str(tool_call['args'])[:100]}...")
    if getattr(message, "name", None) == "task":
        print(f"  <- {str(message.content).strip()[:160]}")

print()
print("Answer:", result["messages"][-1].content)

Step 1: list_files {'path': 'notes'}...
Step 2: read_file {'path': 'notes/2026-08-14-kickoff.md'}...
Step 3: read_file {'path': 'notes/2026-08-21-standup.md'}...
Step 4: read_file {'path': 'notes/2026-08-28-review.md'}...
Step 5: write_todos {'todos': [{'content': 'List files in notes/', 'status': 'completed'}, {'content': 'Read kickoff, st...
Step 6: write_file {'path': 'go-no-go.md', 'content': '# Atlas Go/No-Go — 30 September Beta\n\n**Are we on track?** Yes...
Step 7: task -> fact-checker
  Audit the draft at go-no-go.md for any statements that are contradicted by later notes in the notes/ folder. The source notes are in notes/....
  <- No corrections.

Answer: 
Wrote go-no-go.md answering whether we are on track for the 30 September beta; fact-checker found no corrections.


The trace shows one `task` step where the loop version had nothing. What the
checker read and thought does not appear in the lead's messages at all, because
it happened in a separate conversation.

### Does it actually catch anything?

On a correct draft the checker returns `No corrections.`. So let's write a draft with four known errors — 20 points instead of 15, 2 instead of 4, legal reported as outstanding, and a 17 September checkpoint — and tell the lead to fix nothing itself, so the verdict comes back unedited.

In [ ]:
BAD_DRAFT = '''# Atlas Go/No-Go Assessment

**Are we on track?** Yes, broadly.

- Retrieval has gained **20 points** since the first eval (notes/2026-08-21-standup.md).
- That leaves us **2 points** short of the 80% bar (notes/2026-08-28-review.md).
- **Legal review is still outstanding** and blocks indexing (notes/2026-08-14-kickoff.md).
- The next checkpoint is **17 September** (notes/2026-08-21-standup.md).
'''
print(write_file("draft.md", BAD_DRAFT))

audit = lead.invoke(
    {"messages": [{"role": "user", "content":
        "Do not correct anything yourself and do not write any file. Delegate to the "
        "fact-checker: have it audit draft.md against the notes in notes/, and report "
        "back exactly what it found."}]},
    {"recursion_limit": 40},
)

for message in audit["messages"]:
    if getattr(message, "name", None) == "task":
        print(message.content)
        break

Wrote 399 chars to draft.md



1. "Retrieval has gained **20 points** since the first eval (notes/2026-08-21-standup.md)."
   - Notes actually say: First eval was 61% ("First retrieval eval: 61% of answers cite the correct ticket." from notes/2026-08-21-standup.md), current eval is 76% ("Re-ranking landed. Retrieval eval now at 76%, up from 61%." from notes/2026-08-28-review.md), so gain is 15 points.
   - notes/2026-08-21-standup.md, notes/2026-08-28-review.md

2. "That leaves us **2 points** short of the 80% bar (notes/2026-08-28-review.md)."
   - Notes actually say: Current eval is 76% ("Re-ranking landed. Retrieval eval now at 76%, up from 61%." from notes/2026-08-28-review.md), so 4 points short of 80% bar.
   - notes/2026-08-28-review.md

3. "**Legal review is still outstanding** and blocks indexing (notes/2026-08-14-kickoff.md)."
   - Notes actually say: "Name stripping is live in the ingestion pipeline; legal signed off on 26 Aug." (from notes/2026-08-28-review.md), so legal review is not outstanding.
   - 

### Practice

1. Build the same agent with `backend=FilesystemBackend(...)` and no `tools`
   argument, and compare the traces: the built-in `edit_file` patches
   `go-no-go.md` where our `write_file` rewrites it whole.
2. Give the fact-checker its own `"model"` — a smaller one than the lead — and
   see whether it still catches all four planted errors.